## Import libraries

In [1]:
import os
import re
import gc
import json
import math
import time
import logging
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


## Enviroment

In [2]:
# ---------------------------
# 0) Environment knobs
# ---------------------------
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "0")
os.environ.setdefault("TORCH_CUDNN_V8_API_ENABLED", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("byt-robust")

def print_environment_info():
    logger.info(f"torch={torch.__version__}")
    logger.info(f"cuda_available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        logger.info(f"gpu={torch.cuda.get_device_name(0)}")
        props = torch.cuda.get_device_properties(0)
        logger.info(f"gpu_mem_total_gb={props.total_memory/1024**3:.2f}")
    # BetterTransformer availability (optional)
    try:
        from optimum.bettertransformer import BetterTransformer  # noqa: F401
        logger.info("optimum.bettertransformer is available (will try to use if enabled).")
    except Exception:
        logger.info("optimum.bettertransformer not available (will skip).")

print_environment_info()

2026-03-23 06:25:28,884 | INFO | torch=2.8.0+cu126
2026-03-23 06:25:29,133 | INFO | cuda_available=True
2026-03-23 06:25:29,172 | INFO | gpu=Tesla T4
2026-03-23 06:25:29,172 | INFO | gpu_mem_total_gb=14.56
2026-03-23 06:25:29,173 | INFO | optimum.bettertransformer not available (will skip).


## Configs

In [3]:
@dataclass
class CFG:
    # Paths
    test_path: str = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
    train_path: str = "/kaggle/input/deep-past-initiative-machine-translation/train.csv"
    model_path: str = "/kaggle/input/final-byt5/byt5-akkadian-optimized-34x"
    tokenizer_path: Optional[str] = None
    output_dir: str = "/kaggle/working"

    # Model identity / input formatting
    model_label: str = "base_len115"
    preprocessor_kind: str = "base"
    input_prefix: str = "translate Akkadian to English: "

    # Tokenization / dataloader
    max_length: int = 512
    batch_size: int = 8
    num_workers: int = 2
    pin_memory: bool = True

    # Generation
    num_beams: int = 8
    max_new_tokens: int = 512
    length_penalty: float = 1.3
    early_stopping: bool = True
    no_repeat_ngram_size: int = 0
    repetition_penalty: Optional[float] = None

    # Performance
    use_mixed_precision: bool = True
    use_better_transformer: bool = True
    use_bucket_batching: bool = True
    use_adaptive_beams: bool = True
    use_vectorized_postproc: bool = True

    # Runtime
    checkpoint_freq: int = 2000
    empty_cache_every: int = 10
    validate_samples: int = 6

    # Postprocess
    postprocess_mode: str = "aggressive_only"
    aggressive_trigger_badness: float = 1.5
    min_words_fallback: int = 3

    # Device
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    def __post_init__(self):
        if self.tokenizer_path is None:
            self.tokenizer_path = self.model_path
        if self.device == "cpu":
            self.use_mixed_precision = False
            self.use_better_transformer = False
            self.pin_memory = False



## Preprocessor

Includes

- small gap, big gap convertor
- normalize spaces

In [4]:
# =========================
# CORE NORMALIZATION RULES
# =========================

V2_RE = re.compile(r"([aAeEiIuU])(?:2|₂)")
V3_RE = re.compile(r"([aAeEiIuU])(?:3|₃)")

ACUTE = str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
GRAVE = str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})

GAP_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>|<\s*gap\s*>|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b|\.{3,}|…+|\[\.+\]|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)|\(\s*break\s*\)|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I
)

CHAR_TRANS = str.maketrans({
    "ḫ":"h","Ḫ":"H","ʾ":"",
    "š":"sh","Š":"Sh",
    "ṣ":"s","Ṣ":"S",
    "ṭ":"t","Ṭ":"T",
    "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4",
    "₅":"5","₆":"6","₇":"7","₈":"8","₉":"9",
    "—":"-","–":"-",
})

SUB_X = "ₓ"

DET_UPPER = re.compile(r"\(([A-Z0-9]{1,6})\)")
DET_LOWER = re.compile(r"\(([a-z]{1,4})\)")

WS_RE = re.compile(r"\s+")
FLOAT_RE = re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")

# =========================
# FRACTIONS
# =========================

ALLOWED_FRACS = [
    (1/6,"0.16666"),(1/4,"0.25"),(1/3,"0.33333"),
    (1/2,"0.5"),(2/3,"0.66666"),(3/4,"0.75"),(5/6,"0.83333"),
]

def canon_decimal(x: float) -> str:
    ip = int(math.floor(x + 1e-12))
    frac = x - ip
    best = min(ALLOWED_FRACS, key=lambda t: abs(frac - t[0]))

    if abs(frac - best[0]) <= 2e-3:
        dec = best[1]
        if ip == 0:
            return dec
        return f"{ip}{dec[1:]}"
    return f"{x:.5f}".rstrip("0").rstrip(".")


# =========================
# MAIN UNIFIED CLASS
# =========================

class Preprocessor:
    def __init__(self, use_hints=False):
        self.use_hints = use_hints

        self.OA_HINTS = {
            "KÙ.BABBAR":"silver","AN.NA":"tin","MA.NA":"mina",
            "GÍN":"shekel","TÚG":"textile","GADA":"linen",
            "ŠE":"barley","URUDU":"copper","ANŠE":"donkey",
            "É":"house","DUMU":"son","DUMU.MUNUS":"daughter",
        }

    # ---------- STEP 1: ASCII → DIACRITICS ----------
    def _ascii_to_diacritics(self, s: str) -> str:
        s = s.replace("sz", "š").replace("SZ", "Š")
        s = s.replace("s,", "ṣ").replace("S,", "Ṣ")
        s = s.replace("t,", "ṭ").replace("T,", "Ṭ")

        s = V2_RE.sub(lambda m: m.group(1).translate(ACUTE), s)
        s = V3_RE.sub(lambda m: m.group(1).translate(GRAVE), s)

        return s

    # ---------- STEP 2: MAIN CLEAN ----------
    def _clean(self, t: str) -> str:
        t = self._ascii_to_diacritics(t)

        # determinatives
        t = DET_UPPER.sub(r"\1", t)
        t = DET_LOWER.sub(r"{\1}", t)

        # gaps
        t = GAP_RE.sub(" <gap> ", t)

        # char normalization
        t = t.translate(CHAR_TRANS)
        t = t.replace(SUB_X, "")

        # numbers
        t = FLOAT_RE.sub(lambda m: canon_decimal(float(m.group(1))), t)

        return t

    # ---------- STEP 3: OPTIONAL HINTS ----------
    def _add_hints(self, t: str) -> str:
        if not self.use_hints:
            return t

        found = [v for k, v in self.OA_HINTS.items() if k in t]
        if found:
            t += f" [Commodities: {', '.join(dict.fromkeys(found))}]"
        return t

    # ---------- FINAL PIPELINE ----------
    def preprocess_one(self, text: str) -> str:
        t = str(text or "")

        t = self._clean(t)
        t = self._add_hints(t)

        t = WS_RE.sub(" ", t).strip()
        return t

    def preprocess_batch(self, texts: List[str]) -> List[str]:
        return [self.preprocess_one(t) for t in texts]

## Postprocessor

### Safe Postprocessor

Only contain cleaning format, not editing contents (rewrite sentences)

In [5]:
class SafePostprocessor:
    """
    Conservative cleanup.
    - Preserve common punctuation and formatting.
    - Protect <gap>/<big_gap>.
    - Normalize a few known characters.
    """
    SUBSCRIPT_TO_NORMAL = str.maketrans({
        "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4","₅":"5","₆":"6","₇":"7","₈":"8","₉":"9"
    })

    def __init__(self, use_unicode_fractions: bool = False, strip_dash_old: bool = False):
        self.use_unicode_fractions = use_unicode_fractions
        self.strip_dash_old = strip_dash_old

        self.forbidden_chars = re.compile(r"[⌈⌋⌊⌉]")
        self.multi_space = re.compile(r"\s+")
        self.space_before_punct = re.compile(r"\s+([,.;:!?])")
        self.multi_punct = re.compile(r"([!?.,])\1{2,}")
        self.dashes = re.compile(r"[–—]")

        self.prot_gap = "\uFFF0"
        self.prot_big = "\uFFF1"

        self.bracket_x = re.compile(r"\[\s*x\s*\]|\(\s*x\s*\)", re.IGNORECASE)
        self.bare_x = re.compile(r"(?:(?<=\s)|^)x(?=(\s|$))", re.IGNORECASE)
        self.ellipsis = re.compile(r"(\.\.\.+|…+)")

        self.frac_map = {
            "0.5": "½", "0.25": "¼", "0.75": "¾",
            "1/2": "½", "1/4": "¼", "3/4": "¾",
        }
        self.frac_re = re.compile(r"\b(0\.5|0\.25|0\.75|1/2|1/4|3/4)\b")

    def postprocess_one(self, text: str) -> str:
        if text is None:
            text = ""
        t = str(text)

        t = t.replace("ḫ", "h").replace("Ḫ", "H")
        t = t.translate(self.SUBSCRIPT_TO_NORMAL)
        t = self.dashes.sub("-", t)

        t = self.bracket_x.sub("<gap>", t)
        t = self.bare_x.sub("<gap>", t)
        t = self.ellipsis.sub("<big_gap>", t)

        t = t.replace("<gap>", self.prot_gap).replace("<big_gap>", self.prot_big)
        t = self.forbidden_chars.sub("", t)
        t = t.replace(self.prot_gap, "<gap>").replace(self.prot_big, "<big_gap>")

        if self.strip_dash_old:
            t = t.strip(" -")

        if self.use_unicode_fractions:
            t = self.frac_re.sub(lambda m: self.frac_map.get(m.group(1), m.group(1)), t)

        t = self.space_before_punct.sub(r"\1", t)
        t = self.multi_punct.sub(r"\1", t)
        t = self.multi_space.sub(" ", t).strip()
        return t

    def postprocess_batch(self, texts: List[str]) -> List[str]:
        return [self.postprocess_one(x) for x in texts]


### Aggressive Postprocessor

Use for bad translation (badness >= threshold), include:

- Remove grammatical notes
- Remove repeated words
- Remove repeated phrases (n-grams)
- Consolidate gaps

In [6]:
import re
import pandas as pd
from typing import List, Union

class AggressivePostprocessor:
    def __init__(self, ngram_dedupe_max_n: int = 4):
        self.ngram_dedupe_max_n = ngram_dedupe_max_n

        # 1. Gap and Annotation Patterns
        self.pat_gap = re.compile(r"(\[x\]|\(x\)|\bx\b)", re.I)
        self.pat_big_gap = re.compile(r"(\.{3,}|…|\[\.+\])")
        self.pat_gap_runs = re.compile(r"(?:<gap>\s*){2,}")
        self.pat_biggap_runs = re.compile(r"(?:<big_gap>\s*){2,}")
        self.pat_annotations = re.compile(
            r"\((fem|plur|pl|sing|singular|plural|masc|uncertain|\?|!|damaged|broken)\.?\s*\w*\)", 
            re.I
        )

        # 2. Deduplication and Punctuation Patterns
        self.pat_repeat_word = re.compile(r"\b(\w+)(?:\s+\1\b)+", re.I)
        self.pat_punct_space = re.compile(r"\s+([.,:;!?])")
        self.pat_multi_punct = re.compile(r"([.,!?])\1+")
        self.pat_whitespace = re.compile(r"\s+")

        # 3. Translation Tables
        self.subscript_trans = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
        self.special_chars_trans = str.maketrans("ḫḪ", "hH")
        # Characters to remove (excluding what's inside <gap> tags)
        self.forbidden_chars = '!?()"—–<>⌈⌋⌊⌉[]+ʾ/;'
        self.forbidden_trans = str.maketrans("", "", self.forbidden_chars)

        # 4. Fraction Lookups (Regex, Replacement)
        self.fractions = [
            (re.compile(r"(\d+)\.5\b"), r"\1 ½"), (re.compile(r"\b0\.5\b"), "½"),
            (re.compile(r"(\d+)\.25\b"), r"\1 ¼"), (re.compile(r"\b0\.25\b"), "¼"),
            (re.compile(r"(\d+)\.75\b"), r"\1 ¾"), (re.compile(r"\b0\.75\b"), "¾"),
            (re.compile(r"(\d+)\.33+\d*\b"), r"\1 ⅓"), (re.compile(r"\b0\.33+\d*\b"), "⅓"),
            (re.compile(r"(\d+)\.66+\d*\b"), r"\1 ⅔"), (re.compile(r"\b0\.66+\d*\b"), "⅔"),
        ]

        # 1→1 mappings ONLY
        self.accent_map = str.maketrans({
            "á": "a", "à": "a",
            "é": "e", "è": "e",
            "í": "i", "ì": "i",
            "ú": "u", "ù": "u",
        
            "ā": "a", "ē": "e", "ī": "i", "ū": "u",
            "Ā": "A", "Ē": "E", "Ī": "I", "Ū": "U",
        
            "â": "a", "ê": "e", "î": "i", "û": "u",
            "Â": "A", "Ê": "E", "Î": "I", "Û": "U",
        
            "ä": "a", "ë": "e", "ï": "i", "ü": "u",
            "Ä": "A", "Ë": "E", "Ï": "I", "Ü": "U",
        })
        
        self.akkadian_map = str.maketrans({
            "ṣ": "s",  "Ṣ": "S",
            "ṭ": "t",  "Ṭ": "T",
            "ḫ": "h",  "Ḫ": "H",
            "ḳ": "k",  "Ḳ": "K",
            "ḍ": "d",  "Ḍ": "D",
            "ḇ": "b",  "Ḇ": "B",
            "ḡ": "g",  "Ḡ": "G",
        
            # delete
            "ʾ": None, "ʿ": None,
            "‘": None, "’": None, "'": None, "`": None,
            "º": None, "°": None,
        })

        self.multi_map = {
            # š group
            "š": "sh", "Š": "Sh",
            "ś": "sh", "Ś": "Sh",
        
            # rare composed forms
            "ṣ́": "s", "Ṣ́": "S",
        
            # extra variants
            "ẖ": "h",
        }

    def _apply_multi_map(self, t: str) -> str:
        for k, v in self.multi_map.items():
            t = t.replace(k, v)
        return t

    def _normalize_alphabetical(self, text: str) -> str:
        """
        Strong normalization for Akkadian transliteration.
        """
    
        t = text
    
        # 1. accents
        t = t.translate(self.accent_map)
    
        # 2. 1→1 akkadian
        t = t.translate(self.akkadian_map)
    
        # 3. 1→many (IMPORTANT)
        t = self._apply_multi_map(t)
    
        # 4. subscripts → remove digits
        t = re.sub(r"([a-zA-Z])\d+", r"\1", t)
    
        # 5. determinatives {ki} → ki
        t = re.sub(r"\{([^}]+)\}", r"\1", t)
    
        # 6. CDLI / ORACC
        t = re.sub(r"\bsz\b", "sh", t, flags=re.I)
        t = re.sub(r"\bs,\b", "s", t, flags=re.I)
        t = re.sub(r"\bt,\b", "t", t, flags=re.I)
    
        # 7. remove stray marks
        t = re.sub(r"[ʾ‘']", "", t)
    
        # 8. 🔥 IMPORTANT: fix broken "k rum" → "karum"
        t = re.sub(
            r"\b([bcdfghjklmnpqrstvwxyz])\s+([bcdfghjklmnpqrstvwxyz]{2,})\b",
            r"\1a\2",
            t,
            flags=re.I
        )
    
        # 9. remove junk but keep tags
        t = re.sub(r"[^a-zA-Z<>_ ]+", " ", t)
    
        # 10. normalize spaces
        t = re.sub(r"\s+", " ", t)
    
        return t.strip()

    def _dedupe_ngrams(self, text: str) -> str:
        """Removes recursive loops like 'king of king of' -> 'king of'."""
        tokens = text.split()
        if len(tokens) < 5:
            return text
        
        for n in range(self.ngram_dedupe_max_n, 1, -1):
            i = 0
            while i <= len(tokens) - 2 * n:
                if tokens[i : i + n] == tokens[i + n : i + 2 * n]:
                    del tokens[i + n : i + 2 * n]
                else:
                    i += 1
        return " ".join(tokens)

    def postprocess_one(self, text: str) -> str:
        """The core cleaning logic for a single string."""
        if not isinstance(text, str) or not text.strip():
            return ""

        # Phase 1: Normalization (Subscripts and Special Chars)
        t = text.translate(self.special_chars_trans)
        t = t.translate(self.subscript_trans)

        # Phase 2: Gap Standardization
        t = self.pat_gap.sub("<gap>", t)
        t = self.pat_big_gap.sub("<big_gap>", t)
        t = t.replace("<gap> <gap>", "<big_gap>")
        t = self.pat_gap_runs.sub("<big_gap> ", t)
        t = self.pat_biggap_runs.sub("<big_gap> ", t)

        # Phase 3: Domain Specifics (Fractions & Annotations)
        for pat, repl in self.fractions:
            t = pat.sub(repl, t)
        t = self.pat_annotations.sub("", t)

        # Phase 4: Forbidden Character Removal (Masking Tags)
        # We replace tags with placeholders so translate() doesn't kill the <> or [] in them
        t = t.replace("<gap>", "\x00GAP\x00").replace("<big_gap>", "\x00BIG\x00")
        t = t.translate(self.forbidden_trans)
        t = t.replace("\x00GAP\x00", " <gap> ").replace("\x00BIG\x00", " <big_gap> ")

        # Phase 5: Deduplication (Single words + N-Grams)
        t = self.pat_repeat_word.sub(r"\1", t)
        t = self._dedupe_ngrams(t)

        # Phase 6: Punctuation & Whitespace Cleanup
        t = self.pat_punct_space.sub(r"\1", t)
        t = self.pat_multi_punct.sub(r"\1", t)
        t = self.pat_whitespace.sub(" ", t)
        
        t = t.strip().strip("-").strip()

        t = self._normalize_alphabetical(t)

        return t

    def postprocess_batch(self, translations: List[str]) -> List[str]:
        """
        Process a list of translations. 
        Uses a list comprehension for readability.
        """
        return [self.postprocess_one(t) for t in translations]


## Scoring translated

### Badness score (lower = better)

Include

- empty
- not enough/too many words
- many gaps
- repetitive words
- repetitive phrases

In [7]:
import re

def badness_score(text: str) -> float:
    """Lower is better."""
    if text is None:
        text = ""
        
    if text == "The tablet contains an incomplete inscription.":
        return 10.0
    
    t = str(text).strip()
    if not t:
        return 10.0

    words = t.split()
    n = len(words)

    score = 0.0

    # Length penalties
    if n < 5:
        score += 3.0
    if n < 3:
        score += 3.0
    if n > 500:
        score += 2.0
    if n > 650:
        score += 3.0

    # Gap penalties
    gaps = t.count("<gap>") + t.count("<big_gap>")
    if gaps > 6:
        score += (gaps - 6) * 0.35

    # Repetition penalties
    rep = 0
    for i in range(2, n):
        if words[i].lower() == words[i-1].lower() == words[i-2].lower():
            rep += 1
    score += rep * 0.75

    # Bigram repetitiveness
    if n >= 20:
        bigrams = list(zip(words, words[1:]))
        uniq = len(set(bigrams))
        if uniq > 0:
            repetitiveness = 1.0 - (uniq / max(1, len(bigrams)))
            if repetitiveness > 0.35:
                score += (repetitiveness - 0.35) * 6.0

    # NEW: Non-alphabetical character penalty (small)
    non_alpha_chars = sum(1 for c in t if not c.isalpha() and not c.isspace())
    score += non_alpha_chars * 0.02  # small penalty per char

    return score

### Quality score (higher = better)

Calculates how natrual of the output

Includes:

- Ideal length / Acceptable length
- Capitalized
- Well punctuation
- Include keywords
- Penalize gaps
- Apply badness


In [8]:
# These words are more common in Akkadian translit.
KEYWORDS = {
    "tablet", "king", "city", "god", "silver", "gold", "temple", "house", "palace",
    "year", "son", "daughter", "brother", "mother", "father", "gave", "took", "sent",
    "received", "grain", "sheep", "oil", "wine"
}

In [9]:
def quality_score(text: str) -> float:
    if text is None:
        text = ""

    if text == "The tablet contains an incomplete inscription.":
        return 0.0
    t = str(text).strip()
    
    if not t:
        return 0.0 # so bad

    words = t.split()
    n = len(words)
    score = 0.0
    if 80 <= n <= 120:
        score += 2.0 # +3 if 80 <= len <= 120, check below
    if 50 <= n <= 200:
        score += 1.0 # +1 if 50 <= len <= 200

    # formatting
    if t[0].isupper():
        score += 0.5 # Uppercase the first word check
    if t.endswith((".", "?", "!")):
        score += 0.5 # Sentence ending punctation

    # keyword hints
    lw = {w.strip(".,;:!?\"'()[]").lower() for w in words}
    hit = len(lw.intersection(KEYWORDS))
    score += min(2.0, 0.25 * hit) # +0.25 per keyword, 2 max

    # penalize too many gaps
    gaps = t.count("<gap>") + t.count("<big_gap>")
    score -= min(2.0, 0.15 * gaps) # -0.15 per gap, 2 max

    # penalize heavy repetition by badness
    score -= 0.5 * max(0.0, badness_score(t) - 1.0)
    return score    

### Fuse texts (Final decision)

In [10]:
def fuse_texts(a: str, b: str, tie_badness_thresh=0.5, w_a=0.6, w_b=0.4, prefer="a") -> str:
    """
    Robust fuse:
      1) pick lower badness if clearly better
      2) if close, pick higher weighted quality
    """
    ba = badness_score(a)
    bb = badness_score(b)

    qa = quality_score(a) * w_a
    qb = quality_score(b) * w_b

    if ba + tie_badness_thresh < bb:
        return a
    if bb + tie_badness_thresh < ba:
        return b

    if qa > qb:
        return a
    if qa < qb:
        return b

    return a if prefer=="a" else b

## Dataset

loads dataset + preprocess

In [11]:
class AkkadianDataset(Dataset):
    def __init__(self, df: pd.DataFrame, preprocessor, input_prefix: str = "translate Akkadian to English: "):
        self.ids = df["id"].astype("str").tolist()
        raw = df["transliteration"].astype("object").fillna("").tolist()
        proc = preprocessor.preprocess_batch(raw)
        prefix = input_prefix or ""
        self.inputs = [f"{prefix}{x}" for x in proc]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.ids[i], self.inputs[i]



## Bucket Sampler

Purpose: Merge samples has similar length into 1 batch --> faster inference

In [12]:
class BucketSampler(Sampler[List[int]]):
    def __init__(self, texts: List[str], batch_size: int, num_buckets=32, shuffle=False):
        self.batch_size = batch_size
        self.shuffle = shuffle

        lengths = np.array([max(1, len(t.split())) for t in texts], dtype=np.int32) # Measure text len.
        order = np.argsort(lengths) # Sort by len.
        self.indices = order.tolist()
        
        # Create buckets
        self.num_buckets = max(1, num_buckets)
        self.buckets = np.array_split(self.indices, self.num_buckets)

        # Split into batches
        self.batches = []
        rng = np.random.default_rng(42)
        for b in self.buckets:
            b = list(b)
            if shuffle:
                rng.shuffle(b)
            for i in range(0, len(b), batch_size): # split to batches in every buckets
                chunk = b[i:i+batch_size]
                if len(chunk) > 0:
                    self.batches.append(chunk) # push into self.batches

            if shuffle:
                rng.shuffle(self.batches)

    def __iter__(self):
        # Iteration
        for batch in self.batches:
            yield batch
            
    def __len__(self):
        # return number of batches
        return len(self.batches)


## Inference Engine

Includes

- load translation model
- prepares inputs
- runs generation
- clean the output
- return the prediction

In [13]:
class InferenceEngine:
    def __init__(self, cfg: CFG):
        self.cfg = cfg
        self.device = torch.device(cfg.device)
        self.preprocessor = Preprocessor()

        self.safe_pp = SafePostprocessor(use_unicode_fractions=False, strip_dash_old=False)
        self.safe_pp_stripdash = SafePostprocessor(use_unicode_fractions=False, strip_dash_old=True)
        self.agg_pp = AggressivePostprocessor()

        tokenizer_path = cfg.tokenizer_path or cfg.model_path
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_path).to(self.device)
        self.model.eval()

        if cfg.use_better_transformer and cfg.device == "cuda":
            try:
                from optimum.bettertransformer import BetterTransformer
                self.model = BetterTransformer.transform(self.model)
                logger.info("BetterTransformer enabled.")
            except Exception as e:
                logger.warning(f"BetterTransformer failed, continue without it: {e}")

    def _collate_fn(self, batch):
        ids, texts = zip(*batch)
        enc = self.tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=self.cfg.max_length,
            return_tensors="pt"
        )
        return list(ids), enc

    def _adaptive_beams(self, attention_mask: torch.Tensor) -> int:
        if not self.cfg.use_adaptive_beams:
            return self.cfg.num_beams

        lens = attention_mask.sum(dim=1).detach().cpu().numpy()
        med = float(np.median(lens)) if len(lens) else 0.0
        if med < 100:
            return max(4, self.cfg.num_beams // 2)
        return self.cfg.num_beams

    def _postprocess(self, texts: List[str]) -> List[str]:
        mode = self.cfg.postprocess_mode
        if mode == "safe_only":
            return self.safe_pp.postprocess_batch(texts)
        if mode == "aggressive_only":
            out = self.safe_pp_stripdash.postprocess_batch(texts)
            return self.agg_pp.postprocess_batch(out)

        safe = self.safe_pp.postprocess_batch(texts)
        thr = self.cfg.aggressive_trigger_badness
        refined = []
        for t in safe:
            refined.append(self.agg_pp.postprocess_one(t) if badness_score(t) >= thr else t)
        return refined

    @staticmethod
    def _final_fix_one(t: str, min_words=3) -> str:
        tt = (t or "").strip()
        if not tt:
            return "The tablet contains fragmentary text."

        words = tt.split()
        if len(words) < min_words:
            return "The tablet contains an incomplete inscription."

        if tt and tt[0].isalpha() and tt[0].islower():
            tt = tt[0].upper() + tt[1:]

        if not tt.endswith((".", "!", "?")):
            tt = tt + "."

        tt = re.sub(r"\s+([,.;:!?])", r"", tt)
        tt = re.sub(r"\s+", " ", tt).strip()
        return tt

    def unload(self):
        if hasattr(self, "model"):
            del self.model
        if hasattr(self, "tokenizer"):
            del self.tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def run_inference(self, test_df: pd.DataFrame, run_tag="run") -> pd.DataFrame:
        cfg = self.cfg
        ds = AkkadianDataset(test_df, self.preprocessor, input_prefix=cfg.input_prefix)

        if cfg.use_bucket_batching:
            sampler = BucketSampler(ds.inputs, batch_size=cfg.batch_size, num_buckets=32)
            dl = DataLoader(
                ds,
                batch_sampler=sampler,
                num_workers=cfg.num_workers,
                pin_memory=cfg.pin_memory,
                collate_fn=self._collate_fn
            )
        else:
            dl = DataLoader(
                ds,
                batch_size=cfg.batch_size,
                shuffle=False,
                num_workers=cfg.num_workers,
                pin_memory=cfg.pin_memory,
                collate_fn=self._collate_fn
            )

        results: List[Tuple[str, str]] = []
        t0 = time.time()

        if cfg.use_mixed_precision and cfg.device == "cuda":
            autocast_ctx = torch.cuda.amp.autocast
        else:
            class _NullCtx:
                def __enter__(self): return None
                def __exit__(self, exc_type, exc, tb): return False
            autocast_ctx = lambda: _NullCtx()

        for step, (ids, enc) in enumerate(dl):
            input_ids = enc["input_ids"].to(self.device, non_blocking=True)
            attn = enc["attention_mask"].to(self.device, non_blocking=True)

            beams = self._adaptive_beams(attn)
            gen_kwargs = dict(
                num_beams=beams,
                max_new_tokens=cfg.max_new_tokens,
                length_penalty=cfg.length_penalty,
                early_stopping=cfg.early_stopping,
                no_repeat_ngram_size=cfg.no_repeat_ngram_size
            )
            if cfg.repetition_penalty is not None:
                gen_kwargs["repetition_penalty"] = cfg.repetition_penalty

            with torch.inference_mode():
                with autocast_ctx():
                    out_ids = self.model.generate(
                        input_ids=input_ids,
                        attention_mask=attn,
                        **gen_kwargs
                    )

            decoded = self.tokenizer.batch_decode(out_ids, skip_special_tokens=True)
            processed = self._postprocess(decoded)
            processed = [self._final_fix_one(x, cfg.min_words_fallback) for x in processed]
            results.extend(list(zip(ids, processed)))

            if cfg.checkpoint_freq and (len(results) % cfg.checkpoint_freq == 0):
                ck = pd.DataFrame(results, columns=["id", "translation"])
                ck_path = os.path.join(cfg.output_dir, f"checkpoint_{run_tag}_{len(results)}.csv")
                ck.to_csv(ck_path, index=False)
                logger.info(f"Saved checkpoint: {ck_path}")

            if cfg.device == "cuda" and cfg.empty_cache_every and (step + 1) % cfg.empty_cache_every == 0:
                torch.cuda.empty_cache()

            if (step + 1) % 50 == 0:
                elapsed = time.time() - t0
                logger.info(f"[{run_tag}] step={step+1}/{len(dl)} | done={len(results)} | elapsed={elapsed:.1f}s")

        df_out = pd.DataFrame(results, columns=["id", "translation"])
        self._validate(df_out, run_tag=run_tag)
        return df_out

    def _validate(self, df: pd.DataFrame, run_tag: str):
        if df.empty:
            logger.warning(f"[{run_tag}] Empty output dataframe.")
            return

        lens = df["translation"].fillna("").map(lambda x: len(str(x).split()))
        empties = (df["translation"].fillna("").str.strip() == "").mean() * 100
        short = (lens < 5).sum()

        logger.info(
            f"[{run_tag}] rows={len(df)} | empty%={empties:.2f} | "
            f"len(mean/med/min/max)={lens.mean():.1f}/{lens.median():.1f}/{lens.min()}/{lens.max()} | "
            f"short(<5)={short}"
        )

        k = min(self.cfg.validate_samples, len(df))
        if k > 0:
            sample = df.sample(k, random_state=123)
            for _, r in sample.iterrows():
                logger.info(f"[{run_tag}] sample id={r['id']} | {str(r['translation'])[:160]}")



In [14]:
def make_cfg(base: CFG, preset: str, batch_size: Optional[int]=None, num_beams: Optional[int]=None, length_penalty: Optional[float]=None, repetition_penalty: Optional[float]=None, no_repeat_ngram_size: Optional[int]=None, strip_dash_old: Optional[bool]=None, model_path: Optional[str]=None, tokenizer_path: Optional[str]=None, model_label: Optional[str]=None, preprocessor_kind: Optional[str]=None, input_prefix: Optional[str]=None) -> CFG:
    cfg = CFG(**asdict(base))
    if preset == "baseline":
        cfg.length_penalty = 1.30
        cfg.repetition_penalty = None
    elif preset == "len115":
        cfg.length_penalty = 1.15
        cfg.repetition_penalty = None
    elif preset == "rep_pen":
        cfg.length_penalty = 1.30
        cfg.repetition_penalty = 1.08
    elif preset == "len115_rep":
        cfg.length_penalty = 1.15
        cfg.repetition_penalty = 1.08

    if batch_size is not None:
        cfg.batch_size = batch_size
    if num_beams is not None:
        cfg.num_beams = num_beams
    if length_penalty is not None:
        cfg.length_penalty = length_penalty
    if repetition_penalty is not None:
        cfg.repetition_penalty = repetition_penalty
    if no_repeat_ngram_size is not None:
        cfg.no_repeat_ngram_size = no_repeat_ngram_size
    if model_path is not None:
        cfg.model_path = model_path
    if tokenizer_path is not None:
        cfg.tokenizer_path = tokenizer_path
    else:
        cfg.tokenizer_path = cfg.model_path
    if model_label is not None:
        cfg.model_label = model_label
    if preprocessor_kind is not None:
        cfg.preprocessor_kind = preprocessor_kind
    if input_prefix is not None:
        cfg.input_prefix = input_prefix
    return cfg


def fuse_submissions(df_a: pd.DataFrame, df_b: pd.DataFrame, prefer="a", tie_badness_thresh=0.5, w_a=0.60, w_b=0.40) -> pd.DataFrame:
    a = df_a.set_index("id")["translation"]
    b = df_b.set_index("id")["translation"]
    ids = a.index

    out = []
    for _id in ids:
        ta = a.loc[_id]
        tb = b.loc[_id]
        fused = fuse_texts(
            ta,
            tb,
            prefer=prefer,
            tie_badness_thresh=tie_badness_thresh,
            w_a=w_a,
            w_b=w_b
        )
        out.append((_id, fused))
    return pd.DataFrame(out, columns=["id", "translation"])


def resolve_model_dir(*candidates: str) -> str:
    for cand in candidates:
        if not cand:
            continue
        p = Path(cand)
        if p.is_file() and p.name == "config.json":
            return str(p.parent)
        if p.is_dir():
            if (p / "config.json").exists():
                return str(p)
            nested = sorted(p.rglob("config.json"))
            if nested:
                return str(nested[0].parent)
    raise FileNotFoundError(f"Could not resolve model dir from candidates: {candidates}")



## Helpers to build configs and resolve model paths (three-model ensemble)


## Main: load test, run 3 models, fuse with `fuse_texts`, save individual submissions + final ensemble


In [15]:
base_cfg = CFG()
os.makedirs(base_cfg.output_dir, exist_ok=True)

test_df = pd.read_csv(base_cfg.test_path)
assert "id" in test_df.columns and "transliteration" in test_df.columns, "test.csv must have columns: id, transliteration"

logger.info(f"Loaded test rows={len(test_df)} from {base_cfg.test_path}")
if len(test_df) <= 10:
    logger.warning("Very small test set detected. If this is not intentional, verify test_path points to Kaggle competition test.csv")

model_dirs = {
    "base_len115": resolve_model_dir(
        base_cfg.model_path,
        "/kaggle/input/final-byt5/byt5-akkadian-optimized-34x",
    ),
    "mbr_v2": resolve_model_dir(
        "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1",
        "/kaggle/input/byt5-akkadian-mbr-v2/pytorch/default/1",
        "/kaggle/input/byt5-akkadian-mbr-v2/PyTorch/default/1",
    ),
    "mbr6": resolve_model_dir(
        "/kaggle/input/datasets/spencevanasperdt/mattiaangelibyt5-akkadian-mbrpytorchdefault6",
        "/kaggle/input/spencevanasperdt-mattiaangelibyt5-akkadian-mbrpytorchdefault6",
        "/kaggle/input/mattiaangelibyt5-akkadian-mbrpytorchdefault6/model",
    ),
}

ensemble_cfgs = [
    make_cfg(
        base_cfg,
        preset="len115",
        model_path=model_dirs["base_len115"],
        model_label="base_len115",
        preprocessor_kind="base",
        input_prefix="translate Akkadian to English: ",
    ),
    make_cfg(
        base_cfg,
        preset="baseline",
        model_path=model_dirs["mbr_v2"],
        model_label="mbr_v2",
        preprocessor_kind="mbr_v2",
        input_prefix="",
    ),
    make_cfg(
        base_cfg,
        preset="baseline",
        model_path=model_dirs["mbr6"],
        model_label="mbr6",
        preprocessor_kind="mbr6",
        input_prefix="translate Akkadian to English: ",
    ),
]

with open(os.path.join(base_cfg.output_dir, "run_config_base.json"), "w", encoding="utf-8") as f:
    json.dump({
        "base": asdict(base_cfg),
        "resolved_model_dirs": model_dirs,
        "ensemble": [asdict(cfg) for cfg in ensemble_cfgs],
    }, f, ensure_ascii=False, indent=2)

submissions = {}
for cfg in ensemble_cfgs:
    logger.info(
        f"Run {cfg.model_label}: model={cfg.model_path} | preprocessor={cfg.preprocessor_kind} | "
        f"prefix={repr(cfg.input_prefix)} | len_pen={cfg.length_penalty} | rep_pen={cfg.repetition_penalty}"
    )
    engine = InferenceEngine(cfg)
    sub_df = engine.run_inference(test_df, run_tag=cfg.model_label)
    sub_path = os.path.join(cfg.output_dir, f"submission_{cfg.model_label}.csv")
    sub_df.to_csv(sub_path, index=False)
    logger.info(f"Saved: {sub_path}")
    submissions[cfg.model_label] = sub_df
    engine.unload()
    del engine

fused_12 = fuse_submissions(
    submissions["base_len115"],
    submissions["mbr_v2"],
    prefer="a",
    tie_badness_thresh=0.5,
    w_a=0.58,
    w_b=0.42,
)
fused_12_path = os.path.join(base_cfg.output_dir, "submission_fused_base_mbrv2.csv")
fused_12.to_csv(fused_12_path, index=False)
logger.info(f"Saved: {fused_12_path}")

fused_all = fuse_submissions(
    fused_12,
    submissions["mbr6"],
    prefer="a",
    tie_badness_thresh=0.5,
    w_a=0.55,
    w_b=0.45,
)
fused_all["translation"] = fused_all["translation"].fillna("").map(
    lambda x: InferenceEngine._final_fix_one(str(x), min_words=base_cfg.min_words_fallback)
)

out_path = os.path.join(base_cfg.output_dir, "submission.csv")
fused_all.to_csv(out_path, index=False)
logger.info(f"Saved FINAL submission: {out_path}")

lens = fused_all["translation"].map(lambda x: len(str(x).split()))
logger.info(f"FINAL | rows={len(fused_all)} | len(mean/med/min/max)={lens.mean():.1f}/{lens.median():.1f}/{lens.min()}/{lens.max()}")
print(fused_all.head())
print(fused_all.tail())



2026-03-23 06:25:29,575 | INFO | Loaded test rows=4 from /kaggle/input/deep-past-initiative-machine-translation/test.csv
2026-03-23 06:25:29,576 | WARNING | Very small test set detected. If this is not intentional, verify test_path points to Kaggle competition test.csv
2026-03-23 06:25:29,588 | INFO | Run base_len115: model=/kaggle/input/final-byt5/byt5-akkadian-optimized-34x | preprocessor=base | prefix='translate Akkadian to English: ' | len_pen=1.15 | rep_pen=None
2026-03-23 06:25:33.520136: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774247133.856250      31 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774247134.008763      31 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS wh

  id                                        translation
0  3  I sent our tablet to every single colony and t...
1  0  Thus karum Kanesh say to the <big_gap> datum o...
2  1  As for the tablet of the City you are in the t...
3  2  As soon as you have heard our letter whether h...
  id                                        translation
0  3  I sent our tablet to every single colony and t...
1  0  Thus karum Kanesh say to the <big_gap> datum o...
2  1  As for the tablet of the City you are in the t...
3  2  As soon as you have heard our letter whether h...
